# Quick Data Exploration: MIMIC-IV Sepsis Dataset

A quick exploration of the processed dataset before building the multi-agent model.

**Runtime:** ~5 minutes

**What you'll learn:**
- Dataset size and structure
- Sepsis vs non-sepsis patterns
- Key vital sign differences
- Missing data patterns
- Features most predictive of sepsis

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

# Paths
DATA_PATH = "/content/drive/MyDrive/Sepsis/data/processed/mimic_harmonized"
DATA_FILE = f"{DATA_PATH}/mimic_processed_medium.h5"

print("Libraries loaded!")

## 2. Load Data

In [ ]:
# Load the processed dataset
print("Loading data...")
df = pd.read_hdf(DATA_FILE, key='data')

print(f"\n{'='*60}")
print(f"DATASET OVERVIEW")
print(f"{'='*60}")
print(f"Total observations: {len(df):,}")
print(f"Total patients: {df['subject_id'].nunique()}")
print(f"Total variables: {len(df.columns)}")
print(f"\nPatients with sepsis: {df[df['sepsis_label']==1]['subject_id'].nunique()}")
print(f"Patients without sepsis: {df[df['sepsis_label']==0]['subject_id'].nunique()}")
print(f"Sepsis prevalence: {df[df['sepsis_label']==1]['subject_id'].nunique() / df['subject_id'].nunique() * 100:.1f}%")

In [ ]:
# View columns
print("\nAvailable Variables:")
print("-" * 40)

# Categorize columns
vitals = ['hr', 'resp', 'temp', 'sbp', 'dbp', 'map_value', 'o2sat']
labs = ['creatinine', 'bilirubin', 'platelets', 'lactate', 'glucose', 'bun', 
        'potassium', 'sodium', 'chloride', 'magnesium', 'calcium', 'bicarbonate', 'wbc']
blood_gas = ['pao2', 'paco2', 'ph', 'fio2', 'base_excess', 'ionized_calcium']
labels = ['sepsis_label', 'hours_to_onset', 'sepsis_onset_time']
meta = ['subject_id', 'hadm_id', 'charttime', 'icu_intime', 'icu_outtime']

print(f"\nVital Signs ({len(vitals)}): {vitals}")
print(f"\nLab Values ({len(labs)}): {labs}")
print(f"\nBlood Gas ({len(blood_gas)}): {blood_gas}")
print(f"\nLabels ({len(labels)}): {labels}")
print(f"\nMetadata ({len(meta)}): {meta}")

In [ ]:
# Sample data
print("\nSample Data (first patient):")
sample_patient = df['subject_id'].iloc[0]
sample_data = df[df['subject_id'] == sample_patient].head(10)
display(sample_data[['charttime', 'hr', 'temp', 'sbp', 'map_value', 'o2sat', 'lactate', 'sepsis_label']])

## 3. Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Observation-level labels
label_counts = df['sepsis_label'].value_counts()
colors = ['#3498db', '#e74c3c']
axes[0].bar(['No Sepsis (0)', 'Sepsis (1)'], label_counts.values, color=colors)
axes[0].set_title('Label Distribution (Observations)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# 2. Patient-level sepsis
patient_sepsis = df.groupby('subject_id')['sepsis_label'].max()
patient_counts = patient_sepsis.value_counts()
axes[1].bar(['No Sepsis', 'Sepsis'], patient_counts.values, color=colors)
axes[1].set_title('Patient-Level Sepsis', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Patients')
for i, v in enumerate(patient_counts.values):
    axes[1].text(i, v + 10, f'{v}', ha='center', fontweight='bold')

# 3. Hours to onset distribution
sepsis_data = df[df['sepsis_label'] == 1]
if 'hours_to_onset' in sepsis_data.columns:
    onset_hours = sepsis_data['hours_to_onset'].dropna()
    axes[2].hist(onset_hours, bins=50, color='#2ecc71', edgecolor='black', alpha=0.7)
    axes[2].axvline(x=-6, color='red', linestyle='--', linewidth=2, label='6h prediction window')
    axes[2].set_title('Hours Before Sepsis Onset', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Hours (negative = before onset)')
    axes[2].set_ylabel('Count')
    axes[2].legend()

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/exploration_labels.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nLabel Balance: {label_counts[0]/len(df)*100:.1f}% negative, {label_counts[1]/len(df)*100:.1f}% positive")

## 4. Missing Data Analysis

In [ ]:
# Calculate missing rates for clinical variables
clinical_vars = vitals + labs + blood_gas
missing_rates = df[clinical_vars].isnull().mean().sort_values(ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#2ecc71' if x < 0.3 else '#f39c12' if x < 0.6 else '#e74c3c' for x in missing_rates.values]
bars = ax.barh(missing_rates.index, missing_rates.values * 100, color=colors)

ax.set_xlabel('Missing Rate (%)', fontsize=12)
ax.set_title('Missing Data by Variable', fontsize=14, fontweight='bold')
ax.axvline(x=30, color='orange', linestyle='--', alpha=0.7, label='30% threshold')
ax.axvline(x=60, color='red', linestyle='--', alpha=0.7, label='60% threshold')

# Add percentage labels
for bar, val in zip(bars, missing_rates.values):
    ax.text(val * 100 + 1, bar.get_y() + bar.get_height()/2, 
            f'{val*100:.1f}%', va='center', fontsize=9)

ax.legend(loc='lower right')
ax.set_xlim(0, 105)

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/exploration_missing.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Insights:")
print(f"  - Well-populated (>70%): {list(missing_rates[missing_rates < 0.3].index)}")
print(f"  - Moderately missing (30-60%): {list(missing_rates[(missing_rates >= 0.3) & (missing_rates < 0.6)].index)}")
print(f"  - Highly missing (>60%): {list(missing_rates[missing_rates >= 0.6].index)}")

## 5. Vital Signs: Sepsis vs Non-Sepsis

In [ ]:
# Compare vital signs between sepsis and non-sepsis
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

vital_vars = ['hr', 'resp', 'temp', 'sbp', 'map_value', 'o2sat', 'lactate', 'creatinine']
var_names = ['Heart Rate', 'Respiratory Rate', 'Temperature', 'Systolic BP', 
             'Mean Arterial Pressure', 'O2 Saturation', 'Lactate', 'Creatinine']

for i, (var, name) in enumerate(zip(vital_vars, var_names)):
    if var in df.columns:
        # Get data
        sepsis_vals = df[df['sepsis_label'] == 1][var].dropna()
        nonsepsis_vals = df[df['sepsis_label'] == 0][var].dropna()
        
        # Box plot
        data = [nonsepsis_vals, sepsis_vals]
        bp = axes[i].boxplot(data, labels=['No Sepsis', 'Sepsis'], patch_artist=True)
        bp['boxes'][0].set_facecolor('#3498db')
        bp['boxes'][1].set_facecolor('#e74c3c')
        
        axes[i].set_title(name, fontsize=11, fontweight='bold')
        
        # Add mean values
        axes[i].text(1, nonsepsis_vals.median(), f'median={nonsepsis_vals.median():.1f}', 
                     ha='center', va='bottom', fontsize=8)
        axes[i].text(2, sepsis_vals.median(), f'median={sepsis_vals.median():.1f}', 
                     ha='center', va='bottom', fontsize=8)

plt.suptitle('Vital Signs & Labs: Sepsis vs Non-Sepsis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{DATA_PATH}/exploration_vitals.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical comparison table
print("\n" + "="*80)
print("STATISTICAL COMPARISON: SEPSIS vs NON-SEPSIS")
print("="*80)
print(f"{'Variable':<20} {'Non-Sepsis':>15} {'Sepsis':>15} {'Difference':>15} {'P-value':>12}")
print("-"*80)

for var in vital_vars:
    if var in df.columns:
        sepsis_vals = df[df['sepsis_label'] == 1][var].dropna()
        nonsepsis_vals = df[df['sepsis_label'] == 0][var].dropna()
        
        if len(sepsis_vals) > 0 and len(nonsepsis_vals) > 0:
            # Calculate stats
            ns_mean = nonsepsis_vals.mean()
            s_mean = sepsis_vals.mean()
            diff = s_mean - ns_mean
            diff_pct = (diff / ns_mean) * 100 if ns_mean != 0 else 0
            
            # T-test
            _, p_val = stats.ttest_ind(nonsepsis_vals, sepsis_vals)
            
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
            
            print(f"{var:<20} {ns_mean:>12.2f}    {s_mean:>12.2f}    {diff:>+10.2f} ({diff_pct:>+5.1f}%)  {p_val:>8.2e} {sig}")

print("-"*80)
print("Significance: *** p<0.001, ** p<0.01, * p<0.05")

## 6. Temporal Patterns: How Vitals Change Before Sepsis

In [ ]:
# Analyze how vitals change in hours leading up to sepsis
sepsis_patients = df[df['hours_to_onset'].notna()].copy()

# Bin hours to onset
bins = [-np.inf, -48, -24, -12, -6, -3, 0]
labels_bins = ['>48h', '24-48h', '12-24h', '6-12h', '3-6h', '0-3h']
sepsis_patients['time_bin'] = pd.cut(sepsis_patients['hours_to_onset'], bins=bins, labels=labels_bins)

# Plot temporal trends
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

trend_vars = ['hr', 'resp', 'temp', 'map_value', 'lactate', 'creatinine']
trend_names = ['Heart Rate', 'Respiratory Rate', 'Temperature', 'MAP', 'Lactate', 'Creatinine']

for i, (var, name) in enumerate(zip(trend_vars, trend_names)):
    if var in sepsis_patients.columns:
        # Calculate mean by time bin
        trend_data = sepsis_patients.groupby('time_bin')[var].agg(['mean', 'std']).reset_index()
        trend_data = trend_data.dropna()
        
        if len(trend_data) > 0:
            x_pos = range(len(trend_data))
            axes[i].bar(x_pos, trend_data['mean'], yerr=trend_data['std']/10, 
                       color='#e74c3c', alpha=0.7, capsize=3)
            axes[i].set_xticks(x_pos)
            axes[i].set_xticklabels(trend_data['time_bin'], rotation=45, ha='right')
            axes[i].set_title(f'{name} Before Sepsis', fontsize=11, fontweight='bold')
            axes[i].set_ylabel(name)
            
            # Add trend arrow if increasing/decreasing
            if len(trend_data) >= 2:
                first_val = trend_data['mean'].iloc[0]
                last_val = trend_data['mean'].iloc[-1]
                change = ((last_val - first_val) / first_val) * 100 if first_val != 0 else 0
                arrow = "↑" if change > 5 else "↓" if change < -5 else "→"
                color = 'red' if abs(change) > 10 else 'orange' if abs(change) > 5 else 'gray'
                axes[i].text(0.95, 0.95, f'{arrow} {change:+.1f}%', transform=axes[i].transAxes,
                            ha='right', va='top', fontsize=12, fontweight='bold', color=color)

plt.suptitle('Vital Sign Trends Leading Up to Sepsis Onset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{DATA_PATH}/exploration_temporal.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Insight: Watch for rising HR, Resp, Temp, Lactate and falling MAP before sepsis!")

## 7. Feature Importance Preview

In [ ]:
# Calculate correlation with sepsis label
clinical_vars_available = [v for v in clinical_vars if v in df.columns]
correlations = df[clinical_vars_available + ['sepsis_label']].corr()['sepsis_label'].drop('sepsis_label')
correlations = correlations.dropna().sort_values(key=abs, ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#e74c3c' if x > 0 else '#3498db' for x in correlations.values]
bars = ax.barh(correlations.index, correlations.values, color=colors)

ax.set_xlabel('Correlation with Sepsis Label', fontsize=12)
ax.set_title('Feature Correlation with Sepsis (Absolute Value)', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)

# Add value labels
for bar, val in zip(bars, correlations.values):
    offset = 0.01 if val > 0 else -0.01
    ha = 'left' if val > 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height()/2, 
            f'{val:.3f}', va='center', ha=ha, fontsize=9)

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/exploration_correlations.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 Positive Correlations (higher = more sepsis):")
for var, corr in correlations.head(5).items():
    if corr > 0:
        print(f"  {var}: {corr:.3f}")

print("\nTop 5 Negative Correlations (lower = more sepsis):")
for var, corr in correlations.items():
    if corr < 0:
        print(f"  {var}: {corr:.3f}")

## 8. Sample Patient Timelines

In [ ]:
# Plot timeline for a sepsis patient
sepsis_patient_ids = df[df['sepsis_label'] == 1]['subject_id'].unique()
sample_sepsis_id = sepsis_patient_ids[0]
patient_data = df[df['subject_id'] == sample_sepsis_id].sort_values('charttime').copy()

# Create hour index
patient_data['hour'] = range(len(patient_data))

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()

plot_vars = ['hr', 'temp', 'map_value', 'resp', 'lactate', 'o2sat']
plot_names = ['Heart Rate (bpm)', 'Temperature (°C)', 'MAP (mmHg)', 
              'Respiratory Rate', 'Lactate (mmol/L)', 'O2 Saturation (%)']

# Find sepsis onset hour
onset_idx = patient_data[patient_data['sepsis_label'] == 1].index[0] if any(patient_data['sepsis_label'] == 1) else None
onset_hour = patient_data.loc[onset_idx, 'hour'] if onset_idx else None

for i, (var, name) in enumerate(zip(plot_vars, plot_names)):
    if var in patient_data.columns:
        ax = axes[i]
        
        # Plot values
        ax.plot(patient_data['hour'], patient_data[var], 'b-', linewidth=1.5, marker='o', markersize=3)
        
        # Mark sepsis onset
        if onset_hour is not None:
            ax.axvline(x=onset_hour, color='red', linestyle='--', linewidth=2, label='Sepsis Onset')
            ax.axvspan(onset_hour-6, onset_hour, color='red', alpha=0.1, label='6h Warning Window')
        
        ax.set_xlabel('Hour in ICU')
        ax.set_ylabel(name)
        ax.set_title(name, fontsize=11, fontweight='bold')
        
        if i == 0:
            ax.legend(loc='upper right', fontsize=8)

plt.suptitle(f'Patient {sample_sepsis_id} Timeline (Sepsis Case)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{DATA_PATH}/exploration_timeline.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPatient {sample_sepsis_id}:")
print(f"  Total hours in ICU: {len(patient_data)}")
print(f"  Sepsis onset at hour: {onset_hour}")
print(f"  Warning window: hours {onset_hour-6} to {onset_hour}")

## 9. Summary & Key Takeaways

In [ ]:
print("\n" + "="*70)
print("KEY TAKEAWAYS FOR MODEL BUILDING")
print("="*70)

print("\n1. DATASET SIZE")
print(f"   - {df['subject_id'].nunique()} patients, {len(df):,} observations")
print(f"   - {df[df['sepsis_label']==1]['subject_id'].nunique()} sepsis cases (33% prevalence)")
print(f"   - Good for training a robust model")

print("\n2. CLASS BALANCE")
pos_rate = (df['sepsis_label']==1).mean() * 100
print(f"   - {pos_rate:.1f}% positive labels (fairly balanced)")
print(f"   - May still need class weights or oversampling")

print("\n3. RELIABLE FEATURES (low missing rate)")
reliable = missing_rates[missing_rates < 0.3].index.tolist()
print(f"   - {reliable}")
print(f"   - Use these as primary features")

print("\n4. HIGH-VALUE BUT SPARSE FEATURES")
sparse = missing_rates[(missing_rates >= 0.3) & (missing_rates < 0.7)].index.tolist()
print(f"   - {sparse}")
print(f"   - Include with proper imputation")

print("\n5. MOST PREDICTIVE FEATURES")
top_features = correlations.head(8).index.tolist()
print(f"   - {top_features}")

print("\n6. TEMPORAL PATTERNS")
print("   - HR, Resp, Temp tend to INCREASE before sepsis")
print("   - MAP, O2Sat tend to DECREASE before sepsis")
print("   - Changes visible 6-12 hours before onset")
print("   - Model should capture temporal trends!")

print("\n" + "="*70)
print("READY TO BUILD THE MULTI-AGENT MODEL!")
print("="*70)

In [ ]:
# Save exploration summary
summary = {
    'total_patients': int(df['subject_id'].nunique()),
    'total_observations': len(df),
    'sepsis_patients': int(df[df['sepsis_label']==1]['subject_id'].nunique()),
    'sepsis_prevalence': float(df[df['sepsis_label']==1]['subject_id'].nunique() / df['subject_id'].nunique()),
    'positive_label_rate': float((df['sepsis_label']==1).mean()),
    'reliable_features': reliable,
    'top_predictive_features': top_features,
    'missing_rates': missing_rates.to_dict()
}

import json
with open(f"{DATA_PATH}/exploration_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nExploration summary saved to {DATA_PATH}/exploration_summary.json")
print("\nVisualization files saved:")
print("  - exploration_labels.png")
print("  - exploration_missing.png")
print("  - exploration_vitals.png")
print("  - exploration_temporal.png")
print("  - exploration_correlations.png")
print("  - exploration_timeline.png")

## Next Steps

Now you're ready to build the multi-agent sepsis prediction model!

**Key insights for model design:**
1. Use reliable features (HR, Resp, Temp, BP, O2Sat) as primary inputs
2. Include lab values (Lactate, Creatinine) with proper imputation
3. Model should capture temporal patterns (LSTM/Transformer)
4. Consider class weights due to label imbalance
5. Target: Predict sepsis 6 hours before onset